# Experiment 02 — Temporal Context Features

**Builds on:** Exp01 (Perch v2 frozen embeddings + LR probes)

**New in this experiment:**
1. Deduplicate labels (each window was annotated twice in the CSV — fixed here)
2. Add temporal context features per class: prev-window score, next-window score, file-mean, file-max

**Hypothesis:** A species heard in a neighboring 5-second window is more likely present in the current one. File-level statistics capture recording-site and time-of-day priors.

In [1]:
import gc
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import onnxruntime as ort
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from scipy.special import expit as sigmoid
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DATA_DIR          = Path("../data")
SOUNDSCAPE_DIR    = DATA_DIR / "train_soundscapes"
PERCH_ONNX_PATH   = Path("../data/models/perch/perch_v2_no_dft.onnx")
PERCH_LABELS_PATH = Path("../data/models/perch_tf/assets/labels.csv")
CACHE_DIR         = Path("../data/cache")
CACHE_DIR.mkdir(exist_ok=True)

SR          = 32_000
WIN_SAMPLES = SR * 5
BATCH_SIZE  = 4
N_SPLITS    = 5
PCA_DIM     = 64
MIN_POS     = 3

In [2]:
# ── Taxonomy + Perch label mapping (same as exp01) ───────────────────────────
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
N_CLASSES = len(taxonomy)
taxon_ids    = taxonomy["primary_label"].astype(str).tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}

perch_labels = pd.read_csv(PERCH_LABELS_PATH, header=0)
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))

mapping     = taxonomy.merge(perch_labels, on="scientific_name", how="left")
mapped_mask = mapping["bc_index"].notna()

comp_to_bc = {}
for _, row in mapping[mapped_mask].iterrows():
    comp_idx = taxon_to_idx[str(row["primary_label"])]
    comp_to_bc[comp_idx] = int(row["bc_index"])

print(f"Mapped {mapped_mask.sum()}/{N_CLASSES} species to Perch")

Mapped 203/234 species to Perch


In [3]:
# ── Load + deduplicate labels ────────────────────────────────────────────────
# Each (filename, start) pair appears exactly twice in the CSV (two annotation passes).
# Deduplicate before building features to avoid inflated N and artificially easy CV.
def time_to_sec(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

raw_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
raw_df["start_sec"] = raw_df["start"].apply(time_to_sec)

labels_df = (
    raw_df
    .drop_duplicates(subset=["filename", "start_sec"])
    .sort_values(["filename", "start_sec"])
    .reset_index(drop=True)
)

print(f"Raw rows: {len(raw_df)}  →  Deduplicated: {len(labels_df)} unique windows from {labels_df['filename'].nunique()} files")

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)
print(f"Label matrix: {Y.shape}  |  classes w/ >=1 positive: {(Y.sum(axis=0) > 0).sum()}")

Raw rows: 1478  →  Deduplicated: 739 unique windows from 66 files
Label matrix: (739, 234)  |  classes w/ >=1 positive: 75


In [4]:
# ── Perch inference on deduplicated windows (new cache) ──────────────────────
EMB_CACHE = CACHE_DIR / "exp02_embeddings.npy"
LOG_CACHE = CACHE_DIR / "exp02_logits.npy"

if EMB_CACHE.exists() and LOG_CACHE.exists():
    print("Loading cached Perch outputs...")
    emb_full    = np.load(EMB_CACHE)
    logits_full = np.load(LOG_CACHE)
else:
    print("Running Perch ONNX inference...")
    sess = ort.InferenceSession(str(PERCH_ONNX_PATH), providers=["CPUExecutionProvider"])

    def run_perch(waves):
        out = sess.run(["embedding", "label"], {"inputs": waves})
        return out[0], out[1]

    all_embs, all_logits = [], []
    for fname, grp in tqdm(labels_df.groupby("filename"), desc="Files"):
        audio, _ = sf.read(str(SOUNDSCAPE_DIR / fname), dtype="float32", always_2d=False)
        chunks = []
        for _, row in grp.iterrows():
            start = int(row["start_sec"]) * SR
            chunk = audio[start : start + WIN_SAMPLES]
            if len(chunk) < WIN_SAMPLES:
                chunk = np.pad(chunk, (0, WIN_SAMPLES - len(chunk)))
            chunks.append(chunk.astype(np.float32))

        embs, logits = [], []
        for i in range(0, len(chunks), BATCH_SIZE):
            b = np.stack(chunks[i : i + BATCH_SIZE])
            e, l = run_perch(b)
            embs.append(e); logits.append(l)

        all_embs.append(np.concatenate(embs,   axis=0))
        all_logits.append(np.concatenate(logits, axis=0))
        gc.collect()

    emb_full    = np.concatenate(all_embs,   axis=0).astype(np.float32)
    logits_full = np.concatenate(all_logits, axis=0).astype(np.float32)
    np.save(EMB_CACHE, emb_full)
    np.save(LOG_CACHE, logits_full)
    print("Cached.")

print(f"Embeddings: {emb_full.shape}  |  Logits: {logits_full.shape}")

Running Perch ONNX inference...


Files:   0%|          | 0/66 [00:00<?, ?it/s]

Cached.
Embeddings: (739, 1536)  |  Logits: (739, 14795)


In [5]:
# ── Mapped Perch scores for competition species ──────────────────────────────
scores_raw = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
for comp_idx, bc_idx in comp_to_bc.items():
    scores_raw[:, comp_idx] = logits_full[:, bc_idx]

# ── Build temporal context features ─────────────────────────────────────────
# For each window at position t within its file, compute:
#   prev_score: scores of window at t-1 (zeros if first)
#   next_score: scores of window at t+1 (zeros if last)
#   file_mean : mean scores across all windows in the file
#   file_max  : max  scores across all windows in the file

N = len(labels_df)
prev_scores = np.zeros((N, N_CLASSES), dtype=np.float32)
next_scores = np.zeros((N, N_CLASSES), dtype=np.float32)
mean_scores = np.zeros((N, N_CLASSES), dtype=np.float32)
max_scores  = np.zeros((N, N_CLASSES), dtype=np.float32)

for fname, grp in labels_df.groupby("filename"):
    idx = grp.index.tolist()          # row indices in labels_df (sorted by start_sec)
    file_scores = scores_raw[idx]     # (W, 234)
    fm = file_scores.mean(axis=0)     # (234,)
    fx = file_scores.max(axis=0)      # (234,)

    for pos, row_idx in enumerate(idx):
        mean_scores[row_idx] = fm
        max_scores[row_idx]  = fx
        if pos > 0:
            prev_scores[row_idx] = file_scores[pos - 1]
        if pos < len(idx) - 1:
            next_scores[row_idx] = file_scores[pos + 1]

print("Temporal features built:", prev_scores.shape, next_scores.shape, mean_scores.shape, max_scores.shape)

Temporal features built: (739, 234) (739, 234) (739, 234) (739, 234)


In [6]:
# ── Metric helpers ───────────────────────────────────────────────────────────
def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return np.mean(aps)

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        if 0 < y_true[:, c].sum() < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return (np.mean(aucs) if aucs else 0.0), len(aucs)

In [7]:
# ── OOF: exp01 setup (no temporal features) as fair baseline ─────────────────
# Re-run exp01 logic on deduplicated data so comparison is apples-to-apples.
groups    = labels_df["filename"].values
gkf       = GroupKFold(n_splits=N_SPLITS)
oof_base  = sigmoid(scores_raw).copy()

for fold, (tr, va) in enumerate(gkf.split(emb_full, groups=groups)):
    scaler = StandardScaler()
    pca    = PCA(n_components=PCA_DIM, whiten=True, random_state=42)
    emb_tr = pca.fit_transform(scaler.fit_transform(emb_full[tr]))
    emb_va = pca.transform(scaler.transform(emb_full[va]))
    X_tr = np.concatenate([emb_tr, scores_raw[tr]], axis=1)
    X_va = np.concatenate([emb_va, scores_raw[va]], axis=1)
    active = [c for c in range(N_CLASSES) if Y[tr, c].sum() >= MIN_POS]
    for c in active:
        clf = LogisticRegression(C=0.5, max_iter=1000, solver="lbfgs")
        clf.fit(X_tr, Y[tr, c])
        oof_base[va, c] = clf.predict_proba(X_va)[:, 1]

auc_base, n = macro_auc(Y, oof_base)
cmap_base   = padded_cmap(Y, oof_base)
print(f"Exp01 (dedup, no temporal)  →  AUC: {auc_base:.4f}  |  cmAP: {cmap_base:.4f}")

Exp01 (dedup, no temporal)  →  AUC: 0.8960  |  cmAP: 0.8974


In [8]:
# ── OOF: + temporal context features ────────────────────────────────────────
oof_temp = sigmoid(scores_raw).copy()

for fold, (tr, va) in enumerate(gkf.split(emb_full, groups=groups)):
    scaler = StandardScaler()
    pca    = PCA(n_components=PCA_DIM, whiten=True, random_state=42)
    emb_tr = pca.fit_transform(scaler.fit_transform(emb_full[tr]))
    emb_va = pca.transform(scaler.transform(emb_full[va]))

    # Temporal features must be built from TRAIN-ONLY windows to avoid leakage.
    # Recompute file mean/max using only training windows.
    train_mean = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
    train_max  = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
    for fname, grp in labels_df.iloc[tr].groupby("filename"):
        idx = grp.index.tolist()
        fm = scores_raw[idx].mean(axis=0)
        fx = scores_raw[idx].max(axis=0)
        for i in idx:
            train_mean[i] = fm
            train_max[i]  = fx
    # For val files, use all their windows' scores (no label leakage since these
    # are Perch logits, not Y labels)
    for fname, grp in labels_df.iloc[va].groupby("filename"):
        idx = grp.index.tolist()
        fm = scores_raw[idx].mean(axis=0)
        fx = scores_raw[idx].max(axis=0)
        for i in idx:
            train_mean[i] = fm
            train_max[i]  = fx

    X_tr = np.concatenate([
        emb_tr,
        scores_raw[tr],
        prev_scores[tr],
        next_scores[tr],
        train_mean[tr],
        train_max[tr],
    ], axis=1)
    X_va = np.concatenate([
        emb_va,
        scores_raw[va],
        prev_scores[va],
        next_scores[va],
        train_mean[va],
        train_max[va],
    ], axis=1)

    active = [c for c in range(N_CLASSES) if Y[tr, c].sum() >= MIN_POS]
    for c in active:
        clf = LogisticRegression(C=0.5, max_iter=1000, solver="lbfgs")
        clf.fit(X_tr, Y[tr, c])
        oof_temp[va, c] = clf.predict_proba(X_va)[:, 1]

    auc_f, n_f = macro_auc(Y[va], oof_temp[va])
    print(f"  Fold {fold}: AUC={auc_f:.4f} ({n_f} classes, {len(active)} probes)")

auc_temp, n = macro_auc(Y, oof_temp)
cmap_temp   = padded_cmap(Y, oof_temp)
print(f"\n+ Temporal features  →  AUC: {auc_temp:.4f}  |  cmAP: {cmap_temp:.4f}")

  Fold 0: AUC=0.8319 (32 classes, 60 probes)


  Fold 1: AUC=0.9144 (36 classes, 58 probes)


  Fold 2: AUC=0.8819 (36 classes, 60 probes)


  Fold 3: AUC=0.8402 (44 classes, 56 probes)


  Fold 4: AUC=0.9189 (35 classes, 61 probes)



+ Temporal features  →  AUC: 0.8891  |  cmAP: 0.8946


In [9]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("Experiment 02 Results")
print("=" * 60)
print(f"{'Method':<35} {'Macro AUC':>10} {'Padded cmAP':>12}")
print("-" * 60)
print(f"{'Exp01 (dedup, no temporal)':<35} {auc_base:>10.4f} {cmap_base:>12.4f}")
print(f"{'+ temporal context features':<35} {auc_temp:>10.4f} {cmap_temp:>12.4f}")
print("=" * 60)
print(f"Delta AUC: {auc_temp - auc_base:+.4f}  |  Delta cmAP: {cmap_temp - cmap_base:+.4f}")

Experiment 02 Results
Method                               Macro AUC  Padded cmAP
------------------------------------------------------------
Exp01 (dedup, no temporal)              0.8960       0.8974
+ temporal context features             0.8891       0.8946
Delta AUC: -0.0069  |  Delta cmAP: -0.0028
